In [ ]:
%pip install pandas

In [ ]:
import pandas as pd

file_path = '../data/final_store_data.csv'
df = pd.read_csv(file_path)

df.head()

In [9]:
import pymysql
import pandas as pd
from datetime import datetime

# CSV 파일
file_path = '../data/final_store_data.csv'
df = pd.read_csv(file_path)

# MySQL 연결 설정
connection = pymysql.connect(
    host='localhost',
    port=3307,
    user='root',
    password='1234',
    database='itseats_geo_test',
    charset='utf8mb4'
)

try:
    cursor = connection.cursor()
    
    # 배치 크기
    batch_size = 1000
    total_rows = len(df)
    
    # 1000개씩 배치 처리
    for i in range(0, total_rows, batch_size):
        batch_df = df.iloc[i:i + batch_size]
        
        # parameterized query용 데이터 준비
        values = []
        for _, row in batch_df.iterrows():
            now = datetime.now()
            
            values.append((
                now,  # created_at
                0,    # is_deleted
                now,  # modified_at
                'OPEN',  # business_status
                1000,  # default_delivery_fee
                '테스트 매장',  # description
                row['longitude'],  # longitude for POINT
                row['latitude'],   # latitude for POINT
                3000,  # only_one_delivery_fee
                0,     # orderable
                str(row['도로명전체주소']) if pd.notna(row['도로명전체주소']) else '',
                str(row['사업장명']) if pd.notna(row['사업장명']) else '',
                str(row['소재지전화']) if pd.notna(row['소재지전화']) else '',
                'ACCEPTED',  # store_status
                None,  # franchise_id
                3,  # member_id
                1  # store_category_id
            ))
        
        # 배치 INSERT (executemany 사용)
        query = """
        INSERT INTO `store` (
            `created_at`, `is_deleted`, `modified_at`, `business_status`,
            `default_delivery_fee`, `description`, `location`,
            `only_one_delivery_fee`, `orderable`, `store_address`,
            `store_name`, `store_phone`, `store_status`,
            `franchise_id`, `member_id`, `store_category_id`
        ) VALUES (
            %s, %s, %s, %s, %s, %s, ST_SRID(POINT(%s, %s), 4326),
            %s, %s, %s, %s, %s, %s, %s, %s, %s
        )
        """
        
        cursor.executemany(query, values)
        connection.commit()
        
        print(f"Inserted batch {i//batch_size + 1}: {len(batch_df)} rows (Total: {min(i + batch_size, total_rows)}/{total_rows})")
    
    print("All data inserted successfully!")

except Exception as e:
    connection.rollback()
    print(f"Error: {e}")
    
finally:
    cursor.close()
    connection.close()

Inserted batch 1: 1000 rows (Total: 1000/340068)
Inserted batch 2: 1000 rows (Total: 2000/340068)
Inserted batch 3: 1000 rows (Total: 3000/340068)
Inserted batch 4: 1000 rows (Total: 4000/340068)
Inserted batch 5: 1000 rows (Total: 5000/340068)
Inserted batch 6: 1000 rows (Total: 6000/340068)
Inserted batch 7: 1000 rows (Total: 7000/340068)
Inserted batch 8: 1000 rows (Total: 8000/340068)
Inserted batch 9: 1000 rows (Total: 9000/340068)
Inserted batch 10: 1000 rows (Total: 10000/340068)
Inserted batch 11: 1000 rows (Total: 11000/340068)
Inserted batch 12: 1000 rows (Total: 12000/340068)
Inserted batch 13: 1000 rows (Total: 13000/340068)
Inserted batch 14: 1000 rows (Total: 14000/340068)
Inserted batch 15: 1000 rows (Total: 15000/340068)
Inserted batch 16: 1000 rows (Total: 16000/340068)
Inserted batch 17: 1000 rows (Total: 17000/340068)
Inserted batch 18: 1000 rows (Total: 18000/340068)
Inserted batch 19: 1000 rows (Total: 19000/340068)
Inserted batch 20: 1000 rows (Total: 20000/340068